# hh_rnn3 を Colab の GPU で回す
上から順に実行。ジョブ一覧は `JOBS`、同時に回す数は `PARALLEL`。終わったジョブ（out/<tag>.json がある）は飛ばす。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
D = '/content/drive/MyDrive/hh_rnn3_colab'
R = '/content/repo'
os.makedirs(D + '/out/logs', exist_ok=True)
subprocess.run(f'rm -rf {R} && mkdir -p {R} && cd {R} && unzip -q {D}/bundle.zip', shell=True, check=True)
os.makedirs(R + '/results', exist_ok=True)
if not os.path.exists(R + '/results/hh_rnn3'):
    os.symlink(D + '/out', R + '/results/hh_rnn3')
# 参照（HH の正解）のキャッシュも Drive に置き、2 回目から計算を省く
for f in ('hh_bench_reference.npz', 'hh_bench_val.npz'):
    if not os.path.exists(f'{R}/results/{f}'):
        os.symlink(f'{D}/out/{f}', f'{R}/results/{f}')
subprocess.run('pip -q install efel', shell=True)
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)

In [ ]:
# ジョブの実行部分は Drive の run_jobs.py（colab/run_jobs.py をアップロードしたもの）
JOBS = 'jobs_colab_r1.txt'; PARALLEL = 3; exec(open(f'{D}/run_jobs.py', encoding='utf-8').read())

In [ ]:
# 結果の一覧
import glob, json
for p in sorted(glob.glob(f'{D}/out/*.json')):
    d = json.load(open(p, encoding='utf-8'))
    if 'test' in d:
        print(os.path.basename(p), d['test']['n100'], round(d['test']['gamma'], 3), 'val best', d['best'])